# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a FAIR^2 dataset using the `mlcroissant` library. We will utilize Croissant schema, referencing all dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll access information about the dataset, including its name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print each record set, and then list its available fields, columns and their `@id`s for exploration. All references use the `@id` attribute exclusively.

In [ ]:
# List all record sets and examine their fields and columns by `@id`
record_sets = dataset.metadata.recordSet

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get("field", [])
    columns = rs.get("column", [])
    print("  Fields:")
    for field in fields:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name')}")
    print("  Columns:")
    for col in columns:
        print(f"    - Column @id: {col['@id']}, name: {col.get('name')}")
    print()
# Preview a few records per record set (by @id) as an example
for rs in record_sets:
    print(f"--- Preview Records from RecordSet '{rs['@id']}' ---")
    for rec in dataset.records(record_set=rs['@id']):
        print(rec)
        break  # Show only the first record to avoid excessive output

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the record set and field `@id`s only. DataFrames are indexed by record set `@id`. This enables dynamic referencing of entities for downstream processing.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Preview available columns (referenced by @id) in the first record set
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id:
    print(f"Columns in '{first_record_set_id}':")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate EDA using fields referenced by their `@id`.

In [ ]:
# Example: Choose a numeric field by its @id for analysis
# Below, replace <numeric_field_id> and <group_field_id> with actual field @id from your record set overview

example_record_set_id = first_record_set_id
numeric_field_id = None
group_field_id = None

# Identify numeric fields from the columns for this record set
df = dataframes[example_record_set_id]
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]

# Attempt to find a groupable field (categorical or string)
group_candidates = df.select_dtypes(include='object').columns.tolist()
if group_candidates:
    group_field_id = group_candidates[0]

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using appropriate plots. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id is present, plot group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric or group field found for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset via Croissant schema using `mlcroissant`.
- All data entities, including record sets and fields, were referenced by their `@id`.
- Exploratory analysis and basic visualizations highlighted relationships among numeric and group fields.
- This notebook serves as a reusable template for FAIR^2 dataset exploration and downstream analysis.